# Results Master

Numeric results for the VOTC reorganization study.  
Replaces `09_results.ipynb` + `master_results_table.ipynb`.

**Design:** Anatomical homolog primary, functional secondary.  
Categories always separate. OTC split by resection side. nonOTC cross-sectional only.  
Bootstrap CI from controls (Ayzenberg method). Crawford-Howell for individual patients.  
Excludes sub-017 (polymicrogyria).

In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1: Setup + Helpers
# ═══════════════════════════════════════════════════════════════════════════════

import os, sys
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats as sp_stats
from scipy.stats import ttest_rel, ttest_ind, pearsonr

sys.path.insert(0, '/user_data/csimmon2/git_repos/sym_pt')
from sym_pt_params import processed_dir

BASE     = Path(processed_dir)
SEL_DIR  = BASE / 'group_results' / 'selectivity'
LIU_DIR  = BASE / 'group_results' / 'liu_distinctiveness'
GEO_DIR  = BASE / 'group_results' / 'geometry'
PEAK_DIR = BASE / 'group_results' / 'peak_coords'

COPE_SET   = 'differential'
EXCLUDE    = ['sub-017']
CATEGORIES = ['face', 'house', 'object', 'word']
BILATERAL  = ['house', 'object']
UNILATERAL = ['face', 'word']
PREFERRED_HEMI = {'face': 'right', 'word': 'left', 'house': 'right', 'object': 'left'}

N_ITER = 10_000
RNG    = np.random.default_rng(42)

# ── Helpers ───────────────────────────────────────────────────────────────
def crawford_t(patient_val, ctrl_vals):
    c = np.asarray(ctrl_vals, dtype=float)
    c = c[np.isfinite(c)]
    n = len(c)
    if n < 3: return np.nan, np.nan, n
    m, s = c.mean(), c.std(ddof=1)
    if s == 0: return np.nan, np.nan, n
    t = (patient_val - m) / (s * np.sqrt((n + 1) / n))
    p = 2 * sp_stats.t.sf(abs(t), df=n - 1)
    return t, p, n

def bootstrap_ci(vals, n_draw=None, n_iter=N_ITER, rng=RNG):
    v = np.asarray(vals, dtype=float)
    v = v[np.isfinite(v)]
    n = len(v)
    if n < 3: return np.nan, np.nan
    if n_draw is None: n_draw = n
    n_draw = min(n_draw, n)
    boot = np.array([rng.choice(v, size=n_draw, replace=False).mean() for _ in range(n_iter)])
    return np.percentile(boot, 2.5), np.percentile(boot, 97.5)

def fmt_p(p):
    if np.isnan(p): return '—'
    if p < .001: return f'{p:.4f}***'
    if p < .01:  return f'{p:.3f}**'
    if p < .05:  return f'{p:.3f}*'
    return f'{p:.3f}'

# ── Generic comparison: prints anatomical + functional + Crawford ─────────
def run_comparison(cat, ctrl_lh, ctrl_rh, otc_int_lh, otc_int_rh,
                   patient_info=None, nonotc_lh=None, nonotc_rh=None,
                   higher_is_worse=False):
    """
    Run and print the standard comparison for one category.
    patient_info: list of (sub_id, intact_hemi, value) for Crawford tests.
    Returns dict of results.
    """
    results = {}
    
    # A. ANATOMICAL
    for label, otc_v, ctrl_v, hemi in [
        ('L-res(intRH) v CtrlRH', otc_int_rh, ctrl_rh, 'right'),
        ('R-res(intLH) v CtrlLH', otc_int_lh, ctrl_lh, 'left'),
    ]:
        n_d = max(1, len(otc_v))
        lo, hi = bootstrap_ci(ctrl_v, n_draw=n_d)
        m = np.nanmean(otc_v) if len(otc_v) > 0 else np.nan
        sig = (m < lo or m > hi) if np.isfinite(m) else False
        cm, cs = np.nanmean(ctrl_v), np.nanstd(ctrl_v, ddof=1) if len(ctrl_v) > 1 else np.nan
        sig_str = '*' if sig else 'ns'
        cat_lbl = cat if 'L-res' in label else ''
        print(f'  {cat_lbl:<8} {label:<22} {cm:>7.3f} ({cs:.3f}) n={len(ctrl_v):<3} '
              f'{m:>10.3f} {len(otc_v):>3} [{lo:>9.3f}, {hi:>9.3f}] {sig_str:>6}')
        results[f'anat_{hemi}'] = {'m': m, 'n': len(otc_v), 'ci': (lo, hi), 'sig': sig}
    
    return results

def print_section_header(title):
    print(f'\n{"═"*95}')
    print(title)
    print(f'{"═"*95}')

def print_anat_header():
    print(f'\n── A. ANATOMICAL HOMOLOG (PRIMARY) ──')
    print(f'  {"Cat":<8} {"Comparison":<22} {"Ctrl M(SD)":>16} {"OTC M":>10} {"n":>3} '
          f'{"95% CI":>24} {"Sig":>6}')
    print(f'  {"-"*90}')

print('Setup complete.')

Setup complete.


In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2: Load All Data
# ═══════════════════════════════════════════════════════════════════════════════

# ── Selectivity ───────────────────────────────────────────────────────────
sel_file = SEL_DIR / 'selectivity_summary.csv'
df_sel = pd.read_csv(sel_file)
df_sel = df_sel[~df_sel['sub'].isin(EXCLUDE)]
df_sel['ses_int'] = df_sel['ses'].astype(int)

# First session for cross-sectional
first_ses = df_sel.groupby('sub')['ses_int'].min().reset_index().rename(columns={'ses_int': 'fs'})
df_sel_cs = df_sel.merge(first_ses, on='sub')
df_sel_cs = df_sel_cs[df_sel_cs['ses_int'] == df_sel_cs['fs']].copy()

# ── Liu Distinctiveness ───────────────────────────────────────────────────
liu_file = LIU_DIR / f'liu_distinctiveness_{COPE_SET}.csv'
df_liu = pd.read_csv(liu_file)
df_liu = df_liu[~df_liu['subject_id'].isin(EXCLUDE)]
df_liu = df_liu[df_liu['category'].isin(CATEGORIES)]
df_liu['ses_num'] = pd.to_numeric(df_liu['session'], errors='coerce').astype(int)

first_liu = df_liu.groupby('subject_id')['ses_num'].min().reset_index().rename(columns={'ses_num': 'fs'})
df_liu_cs = df_liu.merge(first_liu, on='subject_id')
df_liu_cs = df_liu_cs[df_liu_cs['ses_num'] == df_liu_cs['fs']].copy()

# ── Geometry Preservation (longitudinal) ──────────────────────────────────
geo_file = GEO_DIR / f'geometry_{COPE_SET}.csv'
df_geo = pd.read_csv(geo_file)
df_geo = df_geo[~df_geo['subject_id'].isin(EXCLUDE)]
df_geo = df_geo[df_geo['category'].isin(CATEGORIES)]

# ── Peak Coords ───────────────────────────────────────────────────────────
peak_file = PEAK_DIR / 'peak_coords.csv'
df_peak = pd.read_csv(peak_file)
df_peak = df_peak[~df_peak['sub'].isin(EXCLUDE)]
df_peak['ses_num'] = pd.to_numeric(df_peak['ses'], errors='coerce').astype(int)

# ── Spatial Drift ─────────────────────────────────────────────────────────
spatial_file = GEO_DIR / f'spatial_{COPE_SET}.csv'
df_spatial = pd.read_csv(spatial_file) if spatial_file.exists() else None
if df_spatial is not None:
    df_spatial = df_spatial[~df_spatial['subject'].str.contains('017')]

# ── MDS Shift ─────────────────────────────────────────────────────────────
mds_file = GEO_DIR / f'mds_{COPE_SET}.csv'
df_mds = pd.read_csv(mds_file) if mds_file.exists() else None
if df_mds is not None:
    df_mds = df_mds[~df_mds['subject'].str.contains('017')]
    df_mds = df_mds[df_mds['category'].isin(CATEGORIES)]

# ── Pairwise Correlations (for RDM Distance) ─────────────────────────────
pair_file = LIU_DIR / f'pairwise_correlations_{COPE_SET}.csv'
df_pw = pd.read_csv(pair_file)
df_pw = df_pw[df_pw['category'].isin(CATEGORIES)]
df_pw = df_pw[~df_pw['subject_id'].isin(EXCLUDE)]
if 'subject' in df_pw.columns:
    df_pw = df_pw[~df_pw['subject'].str.contains('017')]

# ── Extraction helpers ────────────────────────────────────────────────────
def sel_vals(df, cat, hemi):
    """Get values from selectivity-schema: 'left'/'right' for controls,
    'intact_left'/'intact_right' for patients."""
    c = df[df['category'] == cat]
    if hemi in ['left', 'right']:
        return c[c['hemi'] == hemi]
    elif hemi == 'intact_left':
        return c[(c['intact_hemi'] == 'left') & (c['hemi'] == 'left')]
    elif hemi == 'intact_right':
        return c[(c['intact_hemi'] == 'right') & (c['hemi'] == 'right')]
    return c.iloc[0:0]

def geo_vals(df, cat, hemi, vcol):
    """Get values from geo-schema (hemi_label, surgery_side)."""
    c = df[df['category'] == cat]
    if hemi in ['left', 'right']:
        return c[c['hemi_label'] == hemi][vcol].dropna().values
    elif hemi == 'intact_left':
        return c[(c['hemi_label'] == 'intact') & (c['surgery_side'] == 'right')][vcol].dropna().values
    elif hemi == 'intact_right':
        return c[(c['hemi_label'] == 'intact') & (c['surgery_side'] == 'left')][vcol].dropna().values
    return np.array([])

print(f'Controls: {df_sel_cs[df_sel_cs["group"]=="control"]["sub"].nunique()}')
print(f'OTC: {df_sel_cs[df_sel_cs["group"]=="OTC"]["sub"].nunique()}')
print(f'nonOTC: {df_sel_cs[df_sel_cs["group"]=="nonOTC"]["sub"].nunique()}')

Controls: 24
OTC: 16
nonOTC: 9


In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3: Controls Lateralization
# ═══════════════════════════════════════════════════════════════════════════════

ctrl_cs = df_sel_cs[df_sel_cs['group'] == 'control']

for col, label in [('sum_selec_norm', 'Sum Selectivity'),
                    ('mean_act', 'Mean Activation'),
                    ('volume', 'Active Volume')]:
    print(f'\n{"═"*70}')
    print(f'CONTROLS LATERALIZATION: {label}')
    print(f'{"═"*70}')
    for cat in CATEGORIES:
        piv = ctrl_cs[ctrl_cs['category'] == cat][['sub', 'hemi', col]].pivot(
            index='sub', columns='hemi', values=col).dropna()
        if len(piv) < 3: continue
        lh, rh = piv['left'].values, piv['right'].values
        t, p = ttest_rel(lh, rh)
        d = (lh - rh).mean() / (lh - rh).std() if (lh - rh).std() > 0 else np.nan
        direction = 'LH > RH' if (lh - rh).mean() > 0 else 'RH > LH'
        sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
        print(f'  {cat}: LH M={lh.mean():.1f}, RH M={rh.mean():.1f}, '
              f't({len(piv)-1})={t:.3f}, p={p:.4f} {sig}, d={d:.3f} → {direction}')

# Liu
ctrl_liu = df_liu_cs[df_liu_cs['status'] == 'control']
print(f'\n{"═"*70}')
print(f'CONTROLS LATERALIZATION: Liu Distinctiveness')
print(f'{"═"*70}')
for cat in CATEGORIES:
    piv = ctrl_liu[ctrl_liu['category'] == cat][['subject_id', 'hemi_label', 'liu_distinctiveness']].pivot(
        index='subject_id', columns='hemi_label', values='liu_distinctiveness').dropna()
    if len(piv) < 3: continue
    lh, rh = piv['left'].values, piv['right'].values
    t, p = ttest_rel(lh, rh)
    d = (lh - rh).mean() / (lh - rh).std() if (lh - rh).std() > 0 else np.nan
    direction = 'LH > RH' if (lh - rh).mean() > 0 else 'RH > LH'
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f'  {cat}: LH M={lh.mean():.3f}, RH M={rh.mean():.3f}, '
          f't({len(piv)-1})={t:.3f}, p={p:.4f} {sig}, d={d:.3f} → {direction}')


══════════════════════════════════════════════════════════════════════
CONTROLS LATERALIZATION: Sum Selectivity
══════════════════════════════════════════════════════════════════════
  face: LH M=299.4, RH M=493.0, t(23)=-2.914, p=0.0078 **, d=-0.608 → RH > LH
  house: LH M=499.5, RH M=624.9, t(23)=-2.764, p=0.0111 *, d=-0.576 → RH > LH
  object: LH M=1942.0, RH M=1759.7, t(23)=2.450, p=0.0223 *, d=0.511 → LH > RH
  word: LH M=164.3, RH M=29.8, t(23)=3.281, p=0.0033 **, d=0.684 → LH > RH

══════════════════════════════════════════════════════════════════════
CONTROLS LATERALIZATION: Mean Activation
══════════════════════════════════════════════════════════════════════
  face: LH M=4.4, RH M=4.7, t(22)=-1.459, p=0.1587 ns, d=-0.311 → RH > LH
  house: LH M=4.2, RH M=4.2, t(23)=-0.421, p=0.6777 ns, d=-0.088 → RH > LH
  object: LH M=4.7, RH M=4.4, t(23)=4.379, p=0.0002 ***, d=0.913 → LH > RH
  word: LH M=3.4, RH M=2.9, t(22)=3.347, p=0.0029 **, d=0.713 → LH > RH

═════════════════════════

In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4: Cross-Sectional Results — Selectivity + Liu
# ═══════════════════════════════════════════════════════════════════════════════

ctrl_cs = df_sel_cs[df_sel_cs['group'] == 'control']
otc_cs  = df_sel_cs[df_sel_cs['group'] == 'OTC']
non_cs  = df_sel_cs[df_sel_cs['group'] == 'nonOTC']

for metric, label in [('sum_selec_norm', 'Sum Selectivity'),
                       ('mean_act', 'Mean Activation'),
                       ('volume', 'Active Volume')]:
    print_section_header(f'{label} — CROSS-SECTIONAL')
    print_anat_header()
    
    for cat in CATEGORIES:
        c_lh = sel_vals(ctrl_cs, cat, 'left')[metric].dropna().values
        c_rh = sel_vals(ctrl_cs, cat, 'right')[metric].dropna().values
        o_lh = sel_vals(otc_cs, cat, 'intact_left')[metric].dropna().values
        o_rh = sel_vals(otc_cs, cat, 'intact_right')[metric].dropna().values
        run_comparison(cat, c_lh, c_rh, o_lh, o_rh)
    
    # B. Functional
    print(f'\n── B. FUNCTIONAL HOMOLOG (SECONDARY) ──')
    for cat in CATEGORIES:
        pref = PREFERRED_HEMI[cat]
        c_pref = sel_vals(ctrl_cs, cat, pref)[metric].dropna().values
        o_all = np.concatenate([
            sel_vals(otc_cs, cat, 'intact_left')[metric].dropna().values,
            sel_vals(otc_cs, cat, 'intact_right')[metric].dropna().values])
        n_d = max(1, len(o_all))
        lo, hi = bootstrap_ci(c_pref, n_draw=n_d)
        m = np.nanmean(o_all) if len(o_all) > 0 else np.nan
        sig = '*' if (m < lo or m > hi) else 'ns'
        print(f'  {cat:<8} OTC(n={len(o_all)}) v Ctrl {pref}: M={m:.1f}, CI=[{lo:.1f}, {hi:.1f}] {sig}')
    
    # C. nonOTC
    print(f'\n── C. nonOTC (cross-sectional) ──')
    for cat in CATEGORIES:
        n_lh = sel_vals(non_cs, cat, 'intact_left')[metric].dropna().values
        n_rh = sel_vals(non_cs, cat, 'intact_right')[metric].dropna().values
        print(f'  {cat:<8} intactLH: M={np.nanmean(n_lh):>8.1f} (n={len(n_lh)}) | '
              f'intactRH: M={np.nanmean(n_rh):>8.1f} (n={len(n_rh)})')
    
    # Crawford per patient
    print(f'\n── Crawford-Howell per patient (anatomical) ──')
    print(f'  {"Sub":<12} {"Intact":>6} {"Cat":<8} {"Value":>10} {"t":>8} {"p":>10}')
    print(f'  {"-"*60}')
    for sub in sorted(otc_cs['sub'].unique()):
        sd = otc_cs[otc_cs['sub'] == sub]
        intact = sd['intact_hemi'].iloc[0]
        for cat in CATEGORIES:
            val = sd[(sd['category'] == cat) & (sd['hemi'] == intact)][metric].values
            if len(val) == 0: continue
            c_same = sel_vals(ctrl_cs, cat, intact)[metric].dropna().values
            t, p, n = crawford_t(val[0], c_same)
            print(f'  {sub:<12} {intact:>6} {cat:<8} {val[0]:>10.1f} {t:>8.3f} {fmt_p(p):>10}')

# ── Liu Distinctiveness ───────────────────────────────────────────────────
ctrl_liu = df_liu_cs[df_liu_cs['status'] == 'control']
otc_liu  = df_liu_cs[df_liu_cs['group'] == 'OTC']
non_liu  = df_liu_cs[df_liu_cs['group'] == 'nonOTC']
VCOL = 'liu_distinctiveness'

print_section_header(f'LIU DISTINCTIVENESS — CROSS-SECTIONAL')
print_anat_header()

for cat in CATEGORIES:
    c_lh = geo_vals(ctrl_liu, cat, 'left', VCOL)
    c_rh = geo_vals(ctrl_liu, cat, 'right', VCOL)
    o_lh = geo_vals(otc_liu, cat, 'intact_left', VCOL)
    o_rh = geo_vals(otc_liu, cat, 'intact_right', VCOL)
    run_comparison(cat, c_lh, c_rh, o_lh, o_rh)

print(f'\n── B. FUNCTIONAL HOMOLOG (SECONDARY) ──')
for cat in CATEGORIES:
    pref = PREFERRED_HEMI[cat]
    c_pref = geo_vals(ctrl_liu, cat, pref, VCOL)
    o_all = np.concatenate([geo_vals(otc_liu, cat, 'intact_left', VCOL),
                             geo_vals(otc_liu, cat, 'intact_right', VCOL)])
    n_d = max(1, len(o_all))
    lo, hi = bootstrap_ci(c_pref, n_draw=n_d)
    m = np.nanmean(o_all) if len(o_all) > 0 else np.nan
    sig = '*' if (m < lo or m > hi) else 'ns'
    print(f'  {cat:<8} OTC(n={len(o_all)}) v Ctrl {pref}: M={m:.3f}, CI=[{lo:.3f}, {hi:.3f}] {sig}')

print(f'\n── C. nonOTC ──')
for cat in CATEGORIES:
    n_lh = geo_vals(non_liu, cat, 'intact_left', VCOL)
    n_rh = geo_vals(non_liu, cat, 'intact_right', VCOL)
    print(f'  {cat:<8} intactLH: M={np.nanmean(n_lh):.3f} (n={len(n_lh)}) | '
          f'intactRH: M={np.nanmean(n_rh):.3f} (n={len(n_rh)})')

print(f'\n── Crawford-Howell per patient (anatomical) ──')
print(f'  {"Sub":<12} {"Intact":>6} {"Cat":<8} {"Value":>10} {"t":>8} {"p":>10}')
print(f'  {"-"*60}')
for sub in sorted(otc_liu['subject_id'].unique()):
    sd = otc_liu[otc_liu['subject_id'] == sub]
    if sd.empty or 'surgery_side' not in sd.columns: continue
    surgery = sd['surgery_side'].iloc[0]
    intact = 'left' if surgery == 'right' else 'right'
    for cat in CATEGORIES:
        val = sd[(sd['category'] == cat) & (sd['hemi_label'] == 'intact')][VCOL].values
        if len(val) == 0: continue
        c_same = geo_vals(ctrl_liu, cat, intact, VCOL)
        t, p, n = crawford_t(val[0], c_same)
        print(f'  {sub:<12} {intact:>6} {cat:<8} {val[0]:>10.3f} {t:>8.3f} {fmt_p(p):>10}')


═══════════════════════════════════════════════════════════════════════════════════════════════
Sum Selectivity — CROSS-SECTIONAL
═══════════════════════════════════════════════════════════════════════════════════════════════

── A. ANATOMICAL HOMOLOG (PRIMARY) ──
  Cat      Comparison                   Ctrl M(SD)      OTC M   n                   95% CI    Sig
  ------------------------------------------------------------------------------------------
  face     L-res(intRH) v CtrlRH  492.963 (429.661) n=24     386.330   8 [  272.261,   746.982]     ns
           R-res(intLH) v CtrlLH  299.409 (232.089) n=24     444.877   8 [  174.582,   433.262]      *
  house    L-res(intRH) v CtrlRH  624.908 (407.269) n=24     450.796   8 [  402.061,   855.608]     ns
           R-res(intLH) v CtrlLH  499.463 (332.356) n=24     477.169   8 [  321.671,   691.601]     ns
  object   L-res(intRH) v CtrlRH  1759.721 (914.675) n=24    1564.705   8 [ 1244.962,  2270.256]     ns
           R-res(intLH) v C

In [5]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5: Longitudinal Results — Selectivity Change (OTC only)
# ═══════════════════════════════════════════════════════════════════════════════

METRIC_LONG = 'sum_selec_norm'

# ── Identify longitudinal subjects ────────────────────────────────────────
ses_counts = df_sel.groupby('sub')['ses_int'].nunique()
multi_subs = ses_counts[ses_counts >= 2].index.tolist()
df_long = df_sel[df_sel['sub'].isin(multi_subs)].copy()
first_last = df_long.groupby('sub')['ses_int'].agg(['min', 'max']).reset_index()
first_last = first_last[first_last['min'] != first_last['max']]

df_t1 = df_long.merge(first_last[['sub', 'min']], on='sub')
df_t1 = df_t1[df_t1['ses_int'] == df_t1['min']].drop(columns='min')
df_tl = df_long.merge(first_last[['sub', 'max']], on='sub')
df_tl = df_tl[df_tl['ses_int'] == df_tl['max']].drop(columns='max')

ctrl_long = df_t1[df_t1['group'] == 'control']
otc_long  = df_t1[df_t1['group'] == 'OTC']

print(f'Longitudinal controls: {ctrl_long["sub"].nunique()} subjects')
print(f'Longitudinal OTC:      {otc_long["sub"].nunique()} subjects')

# ── Control change scores + bootstrap CIs per cat × hemi ─────────────────
ctrl_change = {}
for cat in CATEGORIES:
    for hemi in ['left', 'right']:
        t1 = df_t1[(df_t1['group'] == 'control') & (df_t1['category'] == cat) &
                    (df_t1['hemi'] == hemi)][['sub', METRIC_LONG]].set_index('sub')
        tl = df_tl[(df_tl['group'] == 'control') & (df_tl['category'] == cat) &
                    (df_tl['hemi'] == hemi)][['sub', METRIC_LONG]].set_index('sub')
        paired = t1.join(tl, lsuffix='_t1', rsuffix='_tl').dropna()
        if len(paired) < 3: continue
        change = (paired[f'{METRIC_LONG}_tl'] - paired[f'{METRIC_LONG}_t1']).values
        ctrl_change[f'{cat}_{hemi}'] = change

print_section_header('CONTROLS: Paired T1 vs T_last')
for key, ch in ctrl_change.items():
    t, p = ttest_rel(np.zeros(len(ch)), ch)
    print(f'  {key}: ΔM={ch.mean():.1f} (SD={ch.std():.1f}), n={len(ch)}, t={t:.3f}, p={p:.4f}')

# Bootstrap
boot_change = {}
for key, changes in ctrl_change.items():
    n_avail = len(changes)
    n_draw = min(4, n_avail)
    boot_change[key] = np.array([RNG.choice(changes, size=n_draw, replace=False).mean()
                                  for _ in range(N_ITER)])

# ── OTC patient change vs control change CI ───────────────────────────────
print_section_header('OTC LONGITUDINAL: Selectivity Change vs Control CI')
print(f'  Anatomical: patient intact hemi change vs same-hemi control change CI')

for sub in sorted(otc_long['sub'].unique()):
    sub_t1 = df_t1[(df_t1['sub'] == sub) & (df_t1['group'] == 'OTC')]
    sub_tl = df_tl[(df_tl['sub'] == sub) & (df_tl['group'] == 'OTC')]
    if sub_t1.empty: continue
    intact = sub_t1['intact_hemi'].iloc[0]
    print(f'\n  {sub} (intact {intact}):')
    for cat in CATEGORIES:
        t1v = sub_t1[(sub_t1['category'] == cat) & (sub_t1['hemi'] == intact)][METRIC_LONG]
        tlv = sub_tl[(sub_tl['category'] == cat) & (sub_tl['hemi'] == intact)][METRIC_LONG]
        if t1v.empty or tlv.empty: continue
        pt_ch = tlv.values[0] - t1v.values[0]
        bk = f'{cat}_{intact}'
        if bk not in boot_change: continue
        ci_lo, ci_hi = np.percentile(boot_change[bk], 2.5), np.percentile(boot_change[bk], 97.5)
        abn = pt_ch < ci_lo or pt_ch > ci_hi
        flag = '** ABNORMAL **' if abn else 'within CI'
        print(f'    {cat:<8}: Δ={pt_ch:>+8.0f}  CI=[{ci_lo:>+8.0f}, {ci_hi:>+8.0f}] {flag}')

Longitudinal controls: 9 subjects
Longitudinal OTC:      5 subjects

═══════════════════════════════════════════════════════════════════════════════════════════════
CONTROLS: Paired T1 vs T_last
═══════════════════════════════════════════════════════════════════════════════════════════════
  face_left: ΔM=42.8 (SD=123.8), n=9, t=-0.978, p=0.3566
  face_right: ΔM=115.5 (SD=168.8), n=9, t=-1.935, p=0.0890
  house_left: ΔM=220.3 (SD=358.7), n=9, t=-1.737, p=0.1206
  house_right: ΔM=274.1 (SD=279.1), n=9, t=-2.778, p=0.0240
  object_left: ΔM=-120.2 (SD=1216.0), n=9, t=0.280, p=0.7868
  object_right: ΔM=-233.0 (SD=1185.1), n=9, t=0.556, p=0.5934
  word_left: ΔM=22.2 (SD=160.3), n=9, t=-0.392, p=0.7051
  word_right: ΔM=9.6 (SD=33.0), n=9, t=-0.824, p=0.4336

═══════════════════════════════════════════════════════════════════════════════════════════════
OTC LONGITUDINAL: Selectivity Change vs Control CI
══════════════════════════════════════════════════════════════════════════════════════════

In [14]:
# ═══════════════════════════════════════════════════════════════════════════════
# ADD-ON CELL: Group-level longitudinal selectivity (anatomical + functional)
# Paste this AFTER the per-patient longitudinal selectivity cell.
# Produces output in the same format as geometry/RDM/etc.
# ═══════════════════════════════════════════════════════════════════════════════

METRICS_LONG_GRP = [('sum_selec_norm', 'Sum Selectivity'),
                    ('mean_act',       'Mean Activation'),
                    ('volume',         'Active Volume')]

PREF_HEMI_GRP = {'face': 'right', 'house': 'right', 'object': 'left', 'word': 'left'}

def get_change_by_subs(sub_list, cat, hemi, metric):
    """Compute T_last - T1 change for a list of subjects, filtering by cat and hemi."""
    changes = []
    for sub in sub_list:
        t1_row = df_t1[(df_t1['sub'] == sub) & (df_t1['category'] == cat) &
                       (df_t1['hemi'] == hemi)]
        tl_row = df_tl[(df_tl['sub'] == sub) & (df_tl['category'] == cat) &
                       (df_tl['hemi'] == hemi)]
        if t1_row.empty or tl_row.empty:
            continue
        t1_val = t1_row[metric].values[0]
        tl_val = tl_row[metric].values[0]
        if np.isnan(t1_val) or np.isnan(tl_val):
            continue
        changes.append(tl_val - t1_val)
    return np.array(changes)


# Get subject lists
ctrl_subs = sorted(df_t1[df_t1['group'] == 'control']['sub'].unique())
otc_subs  = sorted(df_t1[df_t1['group'] == 'OTC']['sub'].unique())

# Split OTC by surgery side
otc_info = df_t1[df_t1['group'] == 'OTC'].groupby('sub').first()
otc_l_subs = sorted(otc_info[otc_info['intact_hemi'] == 'right'].index.tolist())   # L-resection, intact RH
otc_r_subs = sorted(otc_info[otc_info['intact_hemi'] == 'left'].index.tolist())    # R-resection, intact LH

print(f'L-resection (intact RH): {otc_l_subs}')
print(f'R-resection (intact LH): {otc_r_subs}')

for METRIC_LONG, metric_label in METRICS_LONG_GRP:

    print(f'\n{"═"*90}')
    print(f'{metric_label} CHANGE — LONGITUDINAL')
    print(f'{"═"*90}')

    # ── A. ANATOMICAL HOMOLOG ─────────────────────────────────────────────
    print(f'\n── A. ANATOMICAL HOMOLOG (PRIMARY) ──')
    print(f'  {"Cat":<8} {"Comparison":<28} {"Ctrl M(SD)":<22} {"OTC M":>7} {"n":>3} {"95% CI":>25} {"Sig":>6}')
    print(f'  {"-"*90}')

    for cat in CATEGORIES:
        first = True
        for otc_sub_list, intact_hemi, ctrl_hemi, label in [
            (otc_l_subs, 'right', 'right', 'L-res(intRH) v CtrlRH'),
            (otc_r_subs, 'left',  'left',  'R-res(intLH) v CtrlLH'),
        ]:
            # Control change for this hemisphere
            ctrl_ch = get_change_by_subs(ctrl_subs, cat, ctrl_hemi, METRIC_LONG)
            if len(ctrl_ch) < 3:
                continue

            # OTC change for this subgroup
            otc_ch = get_change_by_subs(otc_sub_list, cat, intact_hemi, METRIC_LONG)
            if len(otc_ch) == 0:
                continue

            otc_m = np.mean(otc_ch)
            ctrl_m = np.mean(ctrl_ch)
            ctrl_sd = np.std(ctrl_ch, ddof=1)
            n_otc = len(otc_ch)

            # Bootstrap CI
            n_draw = min(4, len(ctrl_ch))
            boot = np.array([RNG.choice(ctrl_ch, size=n_draw, replace=False).mean()
                             for _ in range(N_ITER)])
            ci_lo, ci_hi = np.percentile(boot, 2.5), np.percentile(boot, 97.5)
            sig = '*' if otc_m < ci_lo or otc_m > ci_hi else 'ns'

            cat_label = cat if first else ''
            first = False
            print(f'  {cat_label:<8} {label:<28} {ctrl_m:>8.1f} ({ctrl_sd:>6.1f}) n={len(ctrl_ch):<3}'
                  f'  {otc_m:>8.1f} {n_otc:>3} [{ci_lo:>10.1f}, {ci_hi:>10.1f}] {sig:>6}')

    # ── B. FUNCTIONAL HOMOLOG ─────────────────────────────────────────────
    print(f'\n── B. FUNCTIONAL HOMOLOG (SECONDARY) ──')

    for cat in CATEGORIES:
        pref = PREF_HEMI_GRP[cat]
        # Control preferred hemisphere change
        ctrl_ch = get_change_by_subs(ctrl_subs, cat, pref, METRIC_LONG)
        if len(ctrl_ch) < 3:
            continue

        # All OTC: each patient's intact hemisphere change
        otc_changes = []
        for sub in otc_subs:
            sub_row = otc_info.loc[sub]
            intact = sub_row['intact_hemi']
            ch = get_change_by_subs([sub], cat, intact, METRIC_LONG)
            if len(ch) > 0:
                otc_changes.append(ch[0])

        if len(otc_changes) == 0:
            continue

        otc_m = np.mean(otc_changes)
        n_otc = len(otc_changes)

        n_draw = min(4, len(ctrl_ch))
        boot = np.array([RNG.choice(ctrl_ch, size=n_draw, replace=False).mean()
                         for _ in range(N_ITER)])
        ci_lo, ci_hi = np.percentile(boot, 2.5), np.percentile(boot, 97.5)
        sig = '*' if otc_m < ci_lo or otc_m > ci_hi else 'ns'

        print(f'  {cat:<8} OTC(n={n_otc}) v Ctrl {pref}: M={otc_m:.1f}, CI=[{ci_lo:.1f}, {ci_hi:.1f}] {sig}')

L-resection (intact RH): ['sub-010', 'sub-021', 'sub-079']
R-resection (intact LH): ['sub-004', 'sub-008']

══════════════════════════════════════════════════════════════════════════════════════════
Sum Selectivity CHANGE — LONGITUDINAL
══════════════════════════════════════════════════════════════════════════════════════════

── A. ANATOMICAL HOMOLOG (PRIMARY) ──
  Cat      Comparison                   Ctrl M(SD)               OTC M   n                    95% CI    Sig
  ------------------------------------------------------------------------------------------
  face     L-res(intRH) v CtrlRH           115.5 ( 179.1) n=9        36.2   3 [     -14.8,      245.0]     ns
           R-res(intLH) v CtrlLH            42.8 ( 131.3) n=9      -207.7   2 [     -62.8,      136.5]      *
  house    L-res(intRH) v CtrlRH           274.1 ( 296.0) n=9      -160.1   3 [      73.6,      505.9]      *
           R-res(intLH) v CtrlLH           220.3 ( 380.4) n=9       -72.6   2 [     -42.4,      489.4]

In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6: Longitudinal — Geometry, RDM Distance, Liu, Peak Drift, Spatial, MDS
# ═══════════════════════════════════════════════════════════════════════════════

# ── Generic longitudinal comparison for geo-schema data ───────────────────
def run_longitudinal_geo(df, vcol, title, ctrl_status='control', otc_group='OTC'):
    """Run anatomical + Crawford for a geo-schema longitudinal measure."""
    ctrl = df[df['status'] == ctrl_status]
    otc  = df[(df['group'] == otc_group) & (df['hemi_label'] == 'intact')]
    
    print_section_header(f'{title} — LONGITUDINAL')
    print_anat_header()
    
    for cat in CATEGORIES:
        c_lh = geo_vals(ctrl, cat, 'left', vcol)
        c_rh = geo_vals(ctrl, cat, 'right', vcol)
        o_lh = geo_vals(otc, cat, 'intact_left', vcol)
        o_rh = geo_vals(otc, cat, 'intact_right', vcol)
        run_comparison(cat, c_lh, c_rh, o_lh, o_rh)
    
    # Functional
    print(f'\n── B. FUNCTIONAL HOMOLOG (SECONDARY) ──')
    for cat in CATEGORIES:
        pref = PREFERRED_HEMI[cat]
        c_pref = geo_vals(ctrl, cat, pref, vcol)
        o_all = np.concatenate([geo_vals(otc, cat, 'intact_left', vcol),
                                 geo_vals(otc, cat, 'intact_right', vcol)])
        n_d = max(1, len(o_all))
        lo, hi = bootstrap_ci(c_pref, n_draw=n_d)
        m = np.nanmean(o_all) if len(o_all) > 0 else np.nan
        sig = '*' if np.isfinite(m) and (m < lo or m > hi) else 'ns'
        print(f'  {cat:<8} OTC(n={len(o_all)}) v Ctrl {pref}: M={m:.3f}, CI=[{lo:.3f}, {hi:.3f}] {sig}')
    
    # Crawford
    print(f'\n── Crawford-Howell per patient (anatomical) ──')
    for sub in sorted(otc['subject_id'].unique()):
        sd = otc[otc['subject_id'] == sub]
        if sd.empty: continue
        surgery = sd['surgery_side'].iloc[0] if 'surgery_side' in sd.columns else 'na'
        intact = 'left' if surgery == 'right' else 'right'
        for cat in CATEGORIES:
            val = sd[sd['category'] == cat][vcol].values
            if len(val) == 0: continue
            c_same = geo_vals(ctrl, cat, intact, vcol)
            t, p, n = crawford_t(val[0], c_same)
            print(f'  {sub:<12} {intact:>6} {cat:<8} {val[0]:>10.3f} {t:>8.3f} {fmt_p(p):>10}')

# ── Geometry Preservation ─────────────────────────────────────────────────
run_longitudinal_geo(df_geo, 'geometry_preservation', 'GEOMETRY PRESERVATION')

# ── RDM Distance (compute first) ─────────────────────────────────────────
ALL_PAIRS = sorted(df_pw['pair'].unique())
ses_c = df_pw.groupby('subject_id')['session'].nunique()
multi = ses_c[ses_c >= 2].index.tolist()
pw_long = df_pw[df_pw['subject_id'].isin(multi)].copy()
pw_long['ses_rank'] = pw_long.groupby('subject_id')['session'].rank(method='dense').astype(int)
max_rank = pw_long.groupby('subject_id')['ses_rank'].transform('max')
pw_long = pw_long[(pw_long['ses_rank'] == 1) | (pw_long['ses_rank'] == max_rank)].copy()
pw_long['tp'] = pw_long['ses_rank'].apply(lambda x: 'T1' if x == 1 else 'T2')

def compute_rdm_distance(df, subject_id, roi_cat):
    sub_df = df[df['subject_id'] == subject_id]
    t1v, t2v = [], []
    for pair in ALL_PAIRS:
        t1 = sub_df[(sub_df['category'] == roi_cat) & (sub_df['pair'] == pair) & (sub_df['tp'] == 'T1')]['fisher_r']
        t2 = sub_df[(sub_df['category'] == roi_cat) & (sub_df['pair'] == pair) & (sub_df['tp'] == 'T2')]['fisher_r']
        if len(t1) > 0 and len(t2) > 0:
            t1v.append(t1.values[0]); t2v.append(t2.values[0])
    if len(t1v) < 6: return np.nan
    return np.sqrt(np.sum((np.array(t1v) - np.array(t2v))**2))

rdm_rows = []
# Controls per hemisphere
ctrl_pw = pw_long[pw_long['status'] == 'control']
for sub in ctrl_pw['subject_id'].unique():
    for hemi in ['left', 'right']:
        sub_h = ctrl_pw[(ctrl_pw['subject_id'] == sub) & (ctrl_pw['hemi_label'] == hemi)]
        if len(sub_h) == 0: continue
        for cat in CATEGORIES:
            d = compute_rdm_distance(sub_h, sub, cat)
            if np.isfinite(d):
                rdm_rows.append({'subject_id': sub, 'group': 'control', 'status': 'control',
                                 'hemi_label': hemi, 'surgery_side': 'na',
                                 'category': cat, 'rdm_distance': d})

# OTC intact hemi
otc_pw = pw_long[(pw_long['group'] == 'OTC') & (pw_long['hemi_label'] == 'intact')]
for sub in otc_pw['subject_id'].unique():
    surgery = otc_pw[otc_pw['subject_id'] == sub]['surgery_side'].iloc[0] if 'surgery_side' in otc_pw.columns else 'na'
    for cat in CATEGORIES:
        d = compute_rdm_distance(otc_pw, sub, cat)
        if np.isfinite(d):
            rdm_rows.append({'subject_id': sub, 'group': 'OTC', 'status': 'patient',
                             'hemi_label': 'intact', 'surgery_side': surgery,
                             'category': cat, 'rdm_distance': d})

df_rdm = pd.DataFrame(rdm_rows)
run_longitudinal_geo(df_rdm, 'rdm_distance', 'RDM DISTANCE')

# ── Liu Distinctiveness longitudinal ──────────────────────────────────────
ses_c_liu = df_liu.groupby('subject_id')['ses_num'].nunique()
multi_liu = ses_c_liu[ses_c_liu >= 2].index.tolist()
liu_long = df_liu[df_liu['subject_id'].isin(multi_liu)].copy()
fl = liu_long.groupby('subject_id')['ses_num'].agg(['min', 'max']).reset_index()
fl = fl[fl['min'] != fl['max']]

liu_t1 = liu_long.merge(fl[['subject_id', 'min']], on='subject_id')
liu_t1 = liu_t1[liu_t1['ses_num'] == liu_t1['min']]
liu_tl = liu_long.merge(fl[['subject_id', 'max']], on='subject_id')
liu_tl = liu_tl[liu_tl['ses_num'] == liu_tl['max']]

# Compute change scores
liu_change_rows = []
for sub in liu_t1['subject_id'].unique():
    for hemi in liu_t1[liu_t1['subject_id'] == sub]['hemi_label'].unique():
        for cat in CATEGORIES:
            t1v = liu_t1[(liu_t1['subject_id'] == sub) & (liu_t1['hemi_label'] == hemi) & 
                          (liu_t1['category'] == cat)]['liu_distinctiveness']
            tlv = liu_tl[(liu_tl['subject_id'] == sub) & (liu_tl['hemi_label'] == hemi) & 
                          (liu_tl['category'] == cat)]['liu_distinctiveness']
            if t1v.empty or tlv.empty: continue
            liu_change_rows.append({**{c: liu_t1[(liu_t1['subject_id']==sub)].iloc[0].get(c, 'na')
                                        for c in ['subject_id', 'group', 'status', 'surgery_side']},
                                    'hemi_label': hemi, 'category': cat,
                                    'delta_liu': tlv.values[0] - t1v.values[0]})
df_liu_delta = pd.DataFrame(liu_change_rows)
if len(df_liu_delta) > 0:
    run_longitudinal_geo(df_liu_delta, 'delta_liu', 'LIU DISTINCTIVENESS CHANGE')


═══════════════════════════════════════════════════════════════════════════════════════════════
GEOMETRY PRESERVATION — LONGITUDINAL
═══════════════════════════════════════════════════════════════════════════════════════════════

── A. ANATOMICAL HOMOLOG (PRIMARY) ──
  Cat      Comparison                   Ctrl M(SD)      OTC M   n                   95% CI    Sig
  ------------------------------------------------------------------------------------------
  face     L-res(intRH) v CtrlRH    0.744 (0.305) n=9        0.871   3 [    0.460,     0.950]     ns
           R-res(intLH) v CtrlLH    0.652 (0.248) n=9        0.504   2 [    0.282,     0.945]     ns
  house    L-res(intRH) v CtrlRH    0.399 (0.528) n=9        0.190   3 [   -0.074,     0.919]     ns
           R-res(intLH) v CtrlLH    0.486 (0.596) n=9       -0.085   2 [   -0.329,     0.973]     ns
  object   L-res(intRH) v CtrlRH    0.637 (0.297) n=9        0.509   3 [    0.377,     0.861]     ns
           R-res(intLH) v CtrlLH   

In [8]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 7: Peak Drift, Spatial Drift, MDS Shift
# ═══════════════════════════════════════════════════════════════════════════════

# ── Peak Drift ────────────────────────────────────────────────────────────
grp_cols = ['sub', 'category', 'hemi']
idx_first = df_peak.groupby(grp_cols)['ses_num'].idxmin()
idx_last  = df_peak.groupby(grp_cols)['ses_num'].idxmax()
pk_t1 = df_peak.loc[idx_first].set_index(grp_cols)
pk_tl = df_peak.loc[idx_last].set_index(grp_cols)
multi_idx = pk_t1.index[pk_t1['ses_num'] != pk_tl.loc[pk_t1.index, 'ses_num']]
pk_t1, pk_tl = pk_t1.loc[multi_idx], pk_tl.loc[multi_idx]

drift = pd.DataFrame(index=multi_idx)
drift['peak_drift_mm'] = np.sqrt(
    (pk_tl['peak_x_mni'] - pk_t1['peak_x_mni'])**2 +
    (pk_tl['peak_y_mni'] - pk_t1['peak_y_mni'])**2 +
    (pk_tl['peak_z_mni'] - pk_t1['peak_z_mni'])**2)
drift['group'] = pk_t1['group']
drift['intact_hemi'] = pk_t1['intact_hemi']
drift = drift.reset_index()

print_section_header(f'PEAK DRIFT — LONGITUDINAL')
print(f'  Controls: {drift[drift["group"]=="control"]["sub"].nunique()}, '
      f'OTC: {drift[drift["group"]=="OTC"]["sub"].nunique()}')
print_anat_header()

for cat in CATEGORIES:
    c_lh = drift[(drift['group'] == 'control') & (drift['category'] == cat) & (drift['hemi'] == 'left')]['peak_drift_mm'].values
    c_rh = drift[(drift['group'] == 'control') & (drift['category'] == cat) & (drift['hemi'] == 'right')]['peak_drift_mm'].values
    o_lh = drift[(drift['group'] == 'OTC') & (drift['category'] == cat) & (drift['intact_hemi'] == 'left') & (drift['hemi'] == 'left')]['peak_drift_mm'].values
    o_rh = drift[(drift['group'] == 'OTC') & (drift['category'] == cat) & (drift['intact_hemi'] == 'right') & (drift['hemi'] == 'right')]['peak_drift_mm'].values
    run_comparison(cat, c_lh, c_rh, o_lh, o_rh)

# Crawford
print(f'\n── Crawford-Howell per patient (anatomical) ──')
for sub in sorted(drift[drift['group'] == 'OTC']['sub'].unique()):
    sd = drift[(drift['sub'] == sub) & (drift['group'] == 'OTC')]
    intact = sd['intact_hemi'].iloc[0]
    for cat in CATEGORIES:
        val = sd[(sd['category'] == cat) & (sd['hemi'] == intact)]['peak_drift_mm'].values
        if len(val) == 0: continue
        c_same = drift[(drift['group'] == 'control') & (drift['category'] == cat) & (drift['hemi'] == intact)]['peak_drift_mm'].values
        t, p, n = crawford_t(val[0], c_same)
        print(f'  {sub:<12} {intact:>6} {cat:<8} {val[0]:>10.2f} {t:>8.3f} {fmt_p(p):>10}')

# ── Spatial Drift ─────────────────────────────────────────────────────────
if df_spatial is not None:
    run_longitudinal_geo(df_spatial, 'relocation_mm', 'SPATIAL DRIFT',
                         ctrl_status='control', otc_group='OTC')

# ── MDS Shift ─────────────────────────────────────────────────────────────
if df_mds is not None:
    otc_mds = df_mds[(df_mds['group'] == 'OTC') & (df_mds['hemi_label'] == 'intact')]
    ctrl_mds = df_mds[df_mds['status'] == 'control']
    
    def mean_mds(df):
        gcols = ['subject', 'measured_category', 'hemi_label', 'group', 'status'] + (
            ['surgery_side'] if 'surgery_side' in df.columns else [])
        return df.groupby(gcols)['mds_shift'].mean().reset_index()
    
    otc_avg = mean_mds(otc_mds)
    ctrl_avg = mean_mds(ctrl_mds)
    
    print_section_header('MDS SHIFT — LONGITUDINAL')
    print(f'  OTC: {otc_avg["subject"].nunique()}, Controls: {ctrl_avg["subject"].nunique()}')
    print_anat_header()
    
    for mcat in CATEGORIES:
        c_lh = ctrl_avg[(ctrl_avg['measured_category'] == mcat) & (ctrl_avg['hemi_label'] == 'left')]['mds_shift'].values
        c_rh = ctrl_avg[(ctrl_avg['measured_category'] == mcat) & (ctrl_avg['hemi_label'] == 'right')]['mds_shift'].values
        o_rh_vals = []; o_lh_vals = []
        for _, row in otc_avg[otc_avg['measured_category'] == mcat].iterrows():
            if 'surgery_side' in otc_avg.columns:
                if row['surgery_side'] == 'left': o_rh_vals.append(row['mds_shift'])
                else: o_lh_vals.append(row['mds_shift'])
        run_comparison(mcat, c_lh, c_rh, np.array(o_lh_vals), np.array(o_rh_vals))


═══════════════════════════════════════════════════════════════════════════════════════════════
PEAK DRIFT — LONGITUDINAL
═══════════════════════════════════════════════════════════════════════════════════════════════
  Controls: 9, OTC: 5

── A. ANATOMICAL HOMOLOG (PRIMARY) ──
  Cat      Comparison                   Ctrl M(SD)      OTC M   n                   95% CI    Sig
  ------------------------------------------------------------------------------------------
  face     L-res(intRH) v CtrlRH    4.502 (9.890) n=9        1.892   3 [    0.355,    11.683]     ns
           R-res(intLH) v CtrlLH    6.076 (6.710) n=9        6.956   2 [    0.918,    17.102]     ns
  house    L-res(intRH) v CtrlRH   10.289 (10.007) n=9       10.728   3 [    2.525,    18.889]     ns
           R-res(intLH) v CtrlLH    7.465 (10.645) n=9       18.802   2 [    0.923,    26.093]     ns
  object   L-res(intRH) v CtrlRH    5.238 (4.338) n=9        3.671   3 [    1.796,     8.938]     ns
           R-res(intLH

In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 8: Supplementary
# ═══════════════════════════════════════════════════════════════════════════════

SUB_ROIS = ['face_FFA', 'face_STS', 'house_PPA', 'house_TOS',
            'object_LOC', 'object_pF', 'word_VWFA', 'word_STG', 'evc']

# ── Sub-ROI: Geometry Preservation ────────────────────────────────────────
geo_full = pd.read_csv(GEO_DIR / f'geometry_{COPE_SET}.csv')
geo_full = geo_full[~geo_full['subject_id'].isin(EXCLUDE)]

print_section_header('SUB-ROI BREAKDOWN: Geometry Preservation')
for cat in SUB_ROIS:
    c = geo_full[geo_full['category'] == cat]
    cv = c[c['status'] == 'control']['geometry_preservation'].dropna()
    ov = c[(c['group'] == 'OTC') & (c['hemi_label'] == 'intact')]['geometry_preservation'].dropna()
    if len(cv) > 0 and len(ov) > 0:
        lo, hi = bootstrap_ci(cv.values, n_draw=max(1, len(ov)))
        sig = '*' if (ov.mean() < lo or ov.mean() > hi) else 'ns'
        print(f'  {cat:<14} Ctrl: {cv.mean():.3f} ({cv.std():.3f}) n={len(cv)}  '
              f'OTC: {ov.mean():.3f} n={len(ov)}  CI=[{lo:.3f}, {hi:.3f}] {sig}')

# ── Sub-ROI: Liu Distinctiveness ──────────────────────────────────────────
liu_full = pd.read_csv(LIU_DIR / f'liu_distinctiveness_{COPE_SET}.csv')
liu_full = liu_full[~liu_full['subject_id'].isin(EXCLUDE)]
liu_full['ses_num'] = pd.to_numeric(liu_full['session'], errors='coerce')
fs = liu_full.groupby('subject_id')['ses_num'].min().reset_index().rename(columns={'ses_num': 'fs'})
liu_cs_f = liu_full.merge(fs, on='subject_id')
liu_cs_f = liu_cs_f[liu_cs_f['ses_num'] == liu_cs_f['fs']]

print_section_header('SUB-ROI BREAKDOWN: Liu Distinctiveness (cross-sectional)')
for cat in SUB_ROIS:
    c = liu_cs_f[liu_cs_f['category'] == cat]
    cv = c[c['status'] == 'control']['liu_distinctiveness'].dropna()
    ov = c[(c['group'] == 'OTC') & (c['hemi_label'] == 'intact')]['liu_distinctiveness'].dropna()
    if len(cv) > 0 and len(ov) > 0:
        lo, hi = bootstrap_ci(cv.values, n_draw=max(1, len(ov)))
        sig = '*' if (ov.mean() < lo or ov.mean() > hi) else 'ns'
        print(f'  {cat:<14} Ctrl: {cv.mean():.3f} ({cv.std():.3f}) n={len(cv)}  '
              f'OTC: {ov.mean():.3f} n={len(ov)}  CI=[{lo:.3f}, {hi:.3f}] {sig}')

# ── Spatial Organization: Peak x-coordinate ───────────────────────────────
pk = df_peak.copy()
first_pk = pk.groupby('sub')['ses_num'].min().reset_index().rename(columns={'ses_num': 'fs'})
pk_cs = pk.merge(first_pk, on='sub')
pk_cs = pk_cs[pk_cs['ses_num'] == pk_cs['fs']]

print_section_header('SPATIAL ORGANIZATION: Peak x-coordinate (medial-lateral)')
for hemi in ['left', 'right']:
    print(f'\n  Hemisphere: {hemi}')
    for cat in CATEGORIES:
        ctrl_x = pk_cs[(pk_cs['group'] == 'control') & (pk_cs['category'] == cat) &
                        (pk_cs['hemi'] == hemi)]['peak_x_mni'].dropna().values
        if len(ctrl_x) < 3: continue
        print(f'    {cat}: Ctrl M={ctrl_x.mean():.1f} (SD={ctrl_x.std():.1f}), n={len(ctrl_x)}')
        for sub in pk_cs[(pk_cs['group'] == 'OTC') & (pk_cs['intact_hemi'] == hemi)]['sub'].unique():
            val = pk_cs[(pk_cs['sub'] == sub) & (pk_cs['category'] == cat) &
                        (pk_cs['hemi'] == hemi)]['peak_x_mni'].values
            if len(val) == 0: continue
            t, p, n = crawford_t(val[0], ctrl_x)
            print(f'      {sub}: x={val[0]:.1f}, t={t:.3f}, p={fmt_p(p)}')

# ── Bilateral vs Unilateral (⚠ REVIEW) ───────────────────────────────────
print_section_header('⚠ BILATERAL vs UNILATERAL (supplementary)')
otc_geo = df_geo[(df_geo['group'] == 'OTC') & (df_geo['hemi_label'] == 'intact')]
uni_per, bi_per = [], []
for sub in sorted(otc_geo['subject_id'].unique()):
    sd = otc_geo[otc_geo['subject_id'] == sub]
    u = sd[sd['category'].isin(UNILATERAL)]['geometry_preservation'].mean()
    b = sd[sd['category'].isin(BILATERAL)]['geometry_preservation'].mean()
    if np.isfinite(u) and np.isfinite(b):
        uni_per.append(u); bi_per.append(b)
        print(f'  {sub}: uni={u:.3f}, bi={b:.3f}, diff={u-b:+.3f}')

if len(uni_per) >= 2:
    t, p = ttest_rel(uni_per, bi_per)
    print(f'\n  Geometry: paired t({len(uni_per)-1})={t:.3f}, p={p:.4f}')
    print(f'  Uni M={np.mean(uni_per):.3f}, Bi M={np.mean(bi_per):.3f}')


═══════════════════════════════════════════════════════════════════════════════════════════════
SUB-ROI BREAKDOWN: Geometry Preservation
═══════════════════════════════════════════════════════════════════════════════════════════════
  face_FFA       Ctrl: 0.706 (0.287) n=18  OTC: 0.745 n=5  CI=[0.479, 0.882] ns
  face_STS       Ctrl: 0.337 (0.480) n=18  OTC: 0.548 n=4  CI=[-0.104, 0.708] ns
  house_PPA      Ctrl: 0.666 (0.377) n=18  OTC: 0.324 n=5  CI=[0.358, 0.873] *
  house_TOS      Ctrl: 0.378 (0.561) n=18  OTC: 0.108 n=5  CI=[-0.036, 0.790] ns
  object_LOC     Ctrl: 0.584 (0.343) n=18  OTC: 0.396 n=5  CI=[0.308, 0.795] ns
  object_pF      Ctrl: 0.624 (0.375) n=18  OTC: 0.676 n=5  CI=[0.325, 0.873] ns
  word_VWFA      Ctrl: 0.517 (0.482) n=12  OTC: 0.139 n=3  CI=[0.033, 0.917] ns
  word_STG       Ctrl: 0.409 (0.425) n=18  OTC: 0.434 n=4  CI=[0.022, 0.735] ns
  evc            Ctrl: 0.222 (0.467) n=16  OTC: 0.004 n=4  CI=[-0.159, 0.616] ns

═══════════════════════════════════════════

In [10]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 9: Confound Checks — Age
# ═══════════════════════════════════════════════════════════════════════════════

try:
    from sym_pt_params import _load_csv
    sub_info = _load_csv()
    first_age = sub_info.groupby('sub')['age'].first().reset_index()
    
    print_section_header('CONFOUND CHECK: Age')
    
    sel_with_age = df_sel_cs.merge(first_age, on='sub')
    for grp in ['control', 'OTC', 'nonOTC']:
        ages = sel_with_age[sel_with_age['group'] == grp].drop_duplicates('sub')['age']
        if len(ages) > 0:
            print(f'  {grp}: M={ages.mean():.1f} (SD={ages.std():.1f}), '
                  f'range={ages.min():.0f}-{ages.max():.0f}, n={len(ages)}')
    
    ctrl_age = sel_with_age[sel_with_age['group'] == 'control'].drop_duplicates('sub')['age'].values
    otc_age  = sel_with_age[sel_with_age['group'] == 'OTC'].drop_duplicates('sub')['age'].values
    if len(ctrl_age) > 2 and len(otc_age) > 2:
        t, p = ttest_ind(ctrl_age, otc_age)
        print(f'\n  Control vs OTC age: t={t:.3f}, p={p:.4f}')
    
    print(f'\n  Age × Sum Selectivity correlation (controls):')
    ctrl_with_age = sel_with_age[sel_with_age['group'] == 'control']
    for cat in CATEGORIES:
        for hemi in ['left', 'right']:
            subset = ctrl_with_age[(ctrl_with_age['category'] == cat) & (ctrl_with_age['hemi'] == hemi)]
            if len(subset) >= 5:
                r, p = pearsonr(subset['age'].values, subset['sum_selec_norm'].values)
                sig = '*' if p < .05 else ''
                print(f'    {cat} {hemi}: r={r:.3f}, p={p:.4f} {sig}')

except (ImportError, AttributeError) as e:
    print(f'Could not load subject info: {e}')
    print('To enable age checks: ensure sym_pt_params._load_csv() is available.')


═══════════════════════════════════════════════════════════════════════════════════════════════
CONFOUND CHECK: Age
═══════════════════════════════════════════════════════════════════════════════════════════════
  control: M=16.1 (SD=7.0), range=8-39, n=24
  OTC: M=16.0 (SD=6.9), range=8-37, n=16
  nonOTC: M=14.8 (SD=2.2), range=10-19, n=9

  Control vs OTC age: t=0.054, p=0.9570

  Age × Sum Selectivity correlation (controls):
    face left: r=0.156, p=0.4657 
    face right: r=0.306, p=0.1458 
    house left: r=-0.242, p=0.2545 
    house right: r=-0.176, p=0.4120 
    object left: r=-0.170, p=0.4273 
    object right: r=-0.107, p=0.6194 
    word left: r=0.016, p=0.9398 
    word right: r=0.013, p=0.9532 


In [16]:
# ═══════════════════════════════════════════════════════════════════════════════
# ADD-ON: Functional homolog for MDS shift and Peak drift (longitudinal)
# Paste after the spatial measures cell. Uses drift, df_mds already in memory.
# ═══════════════════════════════════════════════════════════════════════════════

PREF_HEMI_GRP = {'face': 'right', 'house': 'right', 'object': 'left', 'word': 'left'}

# ═══════════════════════════════════════════════════════════════════════════
# PEAK DRIFT — Functional homolog
# ═══════════════════════════════════════════════════════════════════════════
print('═' * 90)
print('PEAK DRIFT — LONGITUDINAL (Functional Homolog)')
print('═' * 90)
print()
print('── B. FUNCTIONAL HOMOLOG (SECONDARY) ──')

for cat in CATEGORIES:
    pref = PREF_HEMI_GRP[cat]

    # Control: preferred hemisphere
    ctrl_vals = drift[(drift['group'] == 'control') &
                      (drift['category'] == cat) &
                      (drift['hemi'] == pref)]['peak_drift_mm'].dropna().values

    if len(ctrl_vals) < 3:
        print(f'  {cat:<8} insufficient controls')
        continue

    # OTC: each patient's intact hemisphere
    otc_vals = []
    for sub in sorted(drift[drift['group'] == 'OTC']['sub'].unique()):
        sd = drift[(drift['sub'] == sub) & (drift['group'] == 'OTC')]
        if sd.empty:
            continue
        intact = sd['intact_hemi'].iloc[0]
        v = sd[(sd['category'] == cat) & (sd['hemi'] == intact)]['peak_drift_mm'].dropna().values
        if len(v) > 0:
            otc_vals.append(v[0])

    if len(otc_vals) == 0:
        print(f'  {cat:<8} no OTC data')
        continue

    otc_m = np.mean(otc_vals)
    n_otc = len(otc_vals)

    n_draw = min(4, len(ctrl_vals))
    boot = np.array([RNG.choice(ctrl_vals, size=n_draw, replace=False).mean()
                     for _ in range(N_ITER)])
    ci_lo, ci_hi = np.percentile(boot, 2.5), np.percentile(boot, 97.5)
    sig = '*' if otc_m < ci_lo or otc_m > ci_hi else 'ns'

    print(f'  {cat:<8} OTC(n={n_otc}) v Ctrl {pref}: M={otc_m:.1f}, CI=[{ci_lo:.1f}, {ci_hi:.1f}] {sig}')

# ═══════════════════════════════════════════════════════════════════════════
# MDS SHIFT — Functional homolog
# ═══════════════════════════════════════════════════════════════════════════
print()
print('═' * 90)
print('MDS SHIFT — LONGITUDINAL (Functional Homolog)')
print('═' * 90)
print()
print('── B. FUNCTIONAL HOMOLOG (SECONDARY) ──')

if df_mds is not None:
    # Average MDS shift across measured_categories per subject × category ROI
    # Controls: use hemi_label (left/right)
    ctrl_mds = df_mds[df_mds['status'] == 'control'].copy()
    otc_mds = df_mds[(df_mds['group'] == 'OTC') & (df_mds['hemi_label'] == 'intact')].copy()

    # Average across measured_category to get one value per subject × category
    gcols_ctrl = ['subject_id', 'category', 'hemi_label']
    ctrl_avg = ctrl_mds.groupby(gcols_ctrl)['mds_shift'].mean().reset_index()

    gcols_otc = ['subject_id', 'category']
    if 'surgery_side' in otc_mds.columns:
        gcols_otc.append('surgery_side')
    otc_avg = otc_mds.groupby(gcols_otc)['mds_shift'].mean().reset_index()

    # Map preferred hemi to hemi_label for controls
    pref_to_label = {'right': 'right', 'left': 'left'}

    for cat in CATEGORIES:
        pref = PREF_HEMI_GRP[cat]

        # Control preferred hemisphere
        ctrl_vals = ctrl_avg[(ctrl_avg['category'] == cat) &
                             (ctrl_avg['hemi_label'] == pref)]['mds_shift'].dropna().values

        if len(ctrl_vals) < 3:
            print(f'  {cat:<8} insufficient controls')
            continue

        # OTC intact (already filtered to hemi_label == 'intact')
        otc_vals = otc_avg[otc_avg['category'] == cat]['mds_shift'].dropna().values

        if len(otc_vals) == 0:
            print(f'  {cat:<8} no OTC data')
            continue

        otc_m = np.mean(otc_vals)
        n_otc = len(otc_vals)

        n_draw = min(4, len(ctrl_vals))
        boot = np.array([RNG.choice(ctrl_vals, size=n_draw, replace=False).mean()
                         for _ in range(N_ITER)])
        ci_lo, ci_hi = np.percentile(boot, 2.5), np.percentile(boot, 97.5)
        sig = '*' if otc_m < ci_lo or otc_m > ci_hi else 'ns'

        print(f'  {cat:<8} OTC(n={n_otc}) v Ctrl {pref}: M={otc_m:.3f}, CI=[{ci_lo:.3f}, {ci_hi:.3f}] {sig}')
else:
    print('  df_mds not available')

══════════════════════════════════════════════════════════════════════════════════════════
PEAK DRIFT — LONGITUDINAL (Functional Homolog)
══════════════════════════════════════════════════════════════════════════════════════════

── B. FUNCTIONAL HOMOLOG (SECONDARY) ──
  face     OTC(n=5) v Ctrl right: M=3.9, CI=[0.4, 9.2] ns
  house    OTC(n=5) v Ctrl right: M=14.0, CI=[3.1, 18.2] ns
  object   OTC(n=5) v Ctrl left: M=6.4, CI=[2.0, 7.1] ns
  word     OTC(n=5) v Ctrl left: M=11.5, CI=[2.0, 9.6] *

══════════════════════════════════════════════════════════════════════════════════════════
MDS SHIFT — LONGITUDINAL (Functional Homolog)
══════════════════════════════════════════════════════════════════════════════════════════

── B. FUNCTIONAL HOMOLOG (SECONDARY) ──
  face     OTC(n=5) v Ctrl right: M=0.202, CI=[0.104, 0.273] ns
  house    OTC(n=5) v Ctrl right: M=0.288, CI=[0.122, 0.359] ns
  object   OTC(n=5) v Ctrl left: M=0.092, CI=[0.062, 0.212] ns
  word     OTC(n=4) v Ctrl left: M=0.